In [2]:
import pandas as pd
import numpy as np 
from scipy.interpolate import PchipInterpolator
from scipy.optimize import curve_fit
from sklearn.metrics import mean_squared_error
from tensorflow.keras import layers, Model
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Dense, Dropout, LSTM, Input
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec


In [ ]:
df_train_features = pd.read_csv('features/train_features.csv')
df_test_features = pd.read_csv('features/evaluation_features.csv')

In [4]:
del df_train_features['Unnamed: 0']
del df_test_features['Unnamed: 0']

In [5]:
# Calculate the sum of NaNs for each column
nan_counts = df_train_features.isna().sum()

# Filter to keep only the columns that have at least 1 NaN
columns_with_nans = nan_counts[nan_counts > 0]

# Print the result
print(columns_with_nans)


B_load_trend_r2     16
flank_wear_um      120
dtype: int64


In [6]:
# Calculate the sum of NaNs for each column
nan_counts = df_test_features.isna().sum()

# Filter to keep only the columns that have at least 1 NaN
columns_with_nans = nan_counts[nan_counts > 0]

# Print the result
print(columns_with_nans)


B_load_trend_r2    12
flank_wear_um      60
dtype: int64


In [7]:
COLUMNS_TO_DROP = ['B_load_trend_r2']

In [8]:
df_train_features.drop(columns = COLUMNS_TO_DROP, inplace = True, axis = 0)
df_test_features.drop(columns = COLUMNS_TO_DROP, inplace = True, axis = 0)


In [9]:
df_train_features.shape, df_test_features.shape

((156, 318), (77, 318))

# Estimate wear for missing cuts with PCHIP Estimation


To generate dense, per-cut wear labels from the sparse ground-truth measurements 
(provided only at cuts 1, 6, 11, 16, 21, 26), we use **Piecewise Cubic Hermite 
Interpolating Polynomial (PCHIP)** interpolation.

- **Smooth**: No sharp corners between known data points  
- **Monotone**: Never decreases between two increasing measurements (physically correct — tools don't "un-wear")  
- **Preserves Shape**: Does not overshoot or undershoot beyond the range of known values

In [10]:
def pchip_wear_estimation(known_wear_list, cut_no_list=[1, 6, 11, 16, 21, 26]):
    '''
    Piecewise Cubic Hermite Interpolating Polynomial — smooth,
    but preserves monotonicity unlike cubic splines.
    '''
    pchip = PchipInterpolator(cut_no_list, known_wear_list)
    all_cuts = np.arange(1, 27)
    wear_pchip = pchip(all_cuts)
    return wear_pchip  # ✅ numpy array of length 26

In [11]:
cut_data_trainset = pd.read_csv('Training_Dataset/trainset_toolwear_measurement.csv')
cut_data_testset = pd.read_csv('Evaluation_Dataset/evalset_toolwear_measurement.csv')

In [12]:
cut_data_trainset.isna().sum()

Set_No         0
Cut_No         0
Measurement    0
dtype: int64

In [13]:
cut_data_testset.isna().sum()

Set_No         0
Cut_No         0
Measurement    1
dtype: int64

There are missing values of a cut's wear in the evaluation dataset. Therefore, for that cut, we have to estimate the wear for an extra cut. 



In [14]:
cut_data_trainset['Set_No'] = cut_data_trainset['Set_No'].str.extract(r'(\d+)').astype(int)
cut_data_testset['Set_No'] = cut_data_testset['Set_No'].str.extract(r'(\d+)').astype(int)

In [15]:
train_set_no = cut_data_trainset['Set_No'].unique()
test_set_no = cut_data_testset['Set_No'].unique()

In [16]:

# Create an empty list to store the data for each set
all_estimates = []

for set_no in train_set_no:
    # Filter data for the current set
    data = cut_data_trainset[cut_data_trainset['Set_No'] == set_no]
    
    # Get the interpolated wear estimates
    wear_estimate = pchip_wear_estimation(cut_no_list=data['Cut_No'], known_wear_list=data['Measurement'])
    
    # Create a temporary dataframe for this specific set
    temp_df = pd.DataFrame({
        'Set_No': set_no,
        'Cut_No': range(1, len(wear_estimate) + 1),  # Assumes wear_estimate is a list for cuts 1 to 26
        'Wear_Estimate': wear_estimate
    })
    
    # Add it to our master list
    all_estimates.append(temp_df)

# Combine all the individual set dataframes into one large dataset
final_wear_df_trainset = pd.concat(all_estimates, ignore_index=True)

final_wear_df_trainset


,Set_No,Cut_No,Wear_Estimate
0,1,1,31.000000
1,1,2,39.111738
2,1,3,46.815213
3,1,4,54.062820
4,1,5,60.806951
...,...,...,...
151,6,22,125.932800
152,6,23,129.614400
153,6,24,133.929600
154,6,25,138.763200


In [17]:
cut_data_testset.dropna(inplace = True)

In [18]:

# Create an empty list to store the data for each set
all_estimates = []

for set_no in test_set_no:
    # Filter data for the current set
    data = cut_data_testset[cut_data_testset['Set_No'] == set_no]
    
    # Get the interpolated wear estimates
    wear_estimate = pchip_wear_estimation(cut_no_list=data['Cut_No'], known_wear_list=data['Measurement'])
    
    # Create a temporary dataframe for this specific set
    temp_df = pd.DataFrame({
        'Set_No': set_no,
        'Cut_No': range(1, len(wear_estimate) + 1),  # Assumes wear_estimate is a list for cuts 1 to 26
        'Wear_Estimate': wear_estimate
    })
    
    # Add it to our master list
    all_estimates.append(temp_df)

# Combine all the individual set dataframes into one large dataset
final_wear_df_testset = pd.concat(all_estimates, ignore_index=True)

final_wear_df_testset


,Set_No,Cut_No,Wear_Estimate
0,1,1,33.000000
1,1,2,35.687529
2,1,3,38.542588
3,1,4,41.553882
4,1,5,44.710118
...,...,...,...
73,3,22,115.365565
74,3,23,116.646261
75,3,24,117.844174
76,3,25,118.961391


- After estimating the wear for unknown cuts, join it with the original training and the testset for correct index matching with set_no and cut_no

In [19]:

final_wear_df_trainset = final_wear_df_trainset.rename(columns={'Set_No': 'set_no', 'Cut_No': 'cut_no'})

# Perform the left merge
df_train_features = pd.merge(
    df_train_features, 
    final_wear_df_trainset, 
    on=['set_no', 'cut_no'], 
    how='left'
)


In [20]:
final_wear_df_testset = final_wear_df_testset.rename(columns={'Set_No': 'set_no', 'Cut_No': 'cut_no'})

df_test_features = pd.merge(
    df_test_features, 
    final_wear_df_testset, 
    on=['set_no', 'cut_no'], 
    how='left'
)


In [21]:
df_train_features.shape, df_test_features.shape

((156, 319), (77, 319))

In [22]:
df_train_features.drop(columns = ['flank_wear_um'], inplace = True) 
df_test_features.drop(columns = ['flank_wear_um'], inplace = True)

In [23]:
# Calculate the sum of NaNs for each column
nan_counts = df_train_features.isna().sum()

# Filter to keep only the columns that have at least 1 NaN
columns_with_nans = nan_counts[nan_counts > 0]

# Print the result
print(columns_with_nans)


Series([], dtype: int64)


In [24]:
# Calculate the sum of NaNs for each column
nan_counts = df_test_features.isna().sum()

# Filter to keep only the columns that have at least 1 NaN
columns_with_nans = nan_counts[nan_counts > 0]

# Print the result
print(columns_with_nans)


Series([], dtype: int64)


# Feature Selection

- As we have chosen all the 8 bands for sensor feature engineering, lot of features don't show any significance in predicting the target vaule. 
- Therefore, we drop the unwanted columns using the mutual_information_gain for regression
- As we see that there are lot of correlated features too. 


In [25]:
# Calculate correlation of all columns against just the Wear_Estimate
# (Replace 'Wear_Estimate' with whatever your target column is actually named)
target_corr = df_train_features.corr()['Wear_Estimate'].sort_values(ascending=False)

print("--- Top 15 Positively Correlated Features ---")
print(target_corr.head(15))

print("\n--- Top 15 Negatively Correlated Features ---")
print(target_corr.tail(15))


--- Top 15 Positively Correlated Features ---
Wear_Estimate                    1.000000
cut_no                           0.975756
Acceleration_X_g_band4_energy    0.758093
AE_V_band5_energy                0.724725
AE_V_band4_energy                0.722211
AE_V_band2_energy                0.714494
AE_V_band3_energy                0.708296
AE_V_band5_std                   0.691034
AE_V_band4_std                   0.682018
AE_V_band2_std                   0.656160
mainSpndLoad_max                 0.654838
mainSpndLoad_range               0.654838
AE_V_band3_std                   0.654250
AE_V_band1_energy                0.645736
Acceleration_X_g_band4_std       0.630733
Name: Wear_Estimate, dtype: float64

--- Top 15 Negatively Correlated Features ---
X_load_min                        -0.262613
B_load_max                        -0.269532
B_load_std                        -0.328536
AE_V_band2_kurtosis               -0.334858
AE_V_band1_kurtosis               -0.359152
AE_V_band5_kurtosis  

In [26]:
from sklearn.feature_selection import mutual_info_regression
import pandas as pd

MI_THRESHOLD = 0.3

# 1. Calculate MI scores directly on the raw DataFrame
x_for_mi = df_train_features.copy()
X_for_mi = df_train_features.drop(columns=['Wear_Estimate', 'set_no'])
y_for_mi  = df_train_features['Wear_Estimate']

mi_scores = mutual_info_regression(X_for_mi, y_for_mi, random_state=42)

# 2. Create a sorted DataFrame of feature names and their scores
mi_df = pd.DataFrame({'Feature': X_for_mi.columns, 'MI_Score': mi_scores})
mi_df = mi_df.sort_values(by='MI_Score', ascending=False).reset_index(drop=True)

print(f"Cut off for MI Score : {MI_THRESHOLD}\n")
print(mi_df[mi_df['MI_Score'] > MI_THRESHOLD])

# 3. Get the NAMES of features that passed the threshold (not indices!)
selected_feature_names = mi_df[mi_df['MI_Score'] > MI_THRESHOLD]['Feature'].tolist()

print(f"\nTotal features selected: {len(selected_feature_names)}")
print(f"Selected features: {selected_feature_names[:5]} ...")  # Preview first 5

# # 4. Save the feature names so inference can use them too!
# import joblib
# joblib.dump(selected_feature_names, 'model/selected_features.pkl')
# print("Feature names saved to model/selected_features.pkl")


Cut off for MI Score : 0.3

                           Feature  MI_Score
0                           cut_no  1.497425
1    Acceleration_X_g_band4_energy  0.915668
2       Acceleration_X_g_band4_std  0.862698
3       Acceleration_X_g_band3_std  0.760274
4    Acceleration_X_g_band3_energy  0.692915
..                             ...       ...
56  Acceleration_Z_g_band5_entropy  0.323446
57   Acceleration_Z_g_band5_energy  0.314418
58  Acceleration_Z_g_band2_entropy  0.308745
59      Acceleration_Z_g_band5_max  0.308739
60            Acceleration_Z_g_max  0.300519

[61 rows x 2 columns]

Total features selected: 61
Selected features: ['cut_no', 'Acceleration_X_g_band4_energy', 'Acceleration_X_g_band4_std', 'Acceleration_X_g_band3_std', 'Acceleration_X_g_band3_energy'] ...


In [27]:
selected_feature_names.extend(['set_no', 'Wear_Estimate']) # Doing this for train_test split

In [28]:
df_train_features = df_train_features[selected_feature_names]
df_test_features = df_test_features[selected_feature_names]

# Deep Learning Architectures

- LSTMs work well when provided with lot of data, as we have only few data points. 
- We will be utilising ML models like RandomForest, CatBoost, etc.

In [29]:
# def build_lstm_model(timestep, input_dim, output_dim, dropout=0.3, lr=0.001):
#     inp = Input(shape=(timestep, input_dim))
#     out = LSTM(64, return_sequences=True)(inp)
#     out = LSTM(32, return_sequences=False)(out)
#     out = Dropout(dropout)(out)
#     out = Dense(output_dim)(out)
#     model = Model(inputs=inp, outputs=out)
#     model.compile(loss='mae', optimizer=Adam(learning_rate=lr), metrics=['mse'])
#     return model


# def lstm_architecture_2(timestep, input_dim, output_dim, dropout=0.3, lr=0.001):
#     inp = Input(shape=(timestep, input_dim))
#     out = LSTM(64, return_sequences=True)(inp)
#     out = LSTM(64, return_sequences=True)(out)
#     out = LSTM(64, return_sequences=False)(out)
#     out = Dropout(dropout)(out)
#     out = Dense(output_dim)(out)
#     model = Model(inputs=inp, outputs=out)
#     model.compile(loss='mae', optimizer=Adam(learning_rate=lr), metrics=['mse'])
#     return model


# def build_mlp(input_dim, dropout=0.3, lr=0.001):
#     inp = layers.Input(shape=(input_dim,), name='features')
#     x   = layers.Dense(256, activation='relu')(inp)
#     x   = layers.BatchNormalization()(x)
#     x   = layers.Dropout(dropout)(x)
#     x   = layers.Dense(128, activation='relu')(x)
#     x   = layers.BatchNormalization()(x)
#     x   = layers.Dropout(dropout)(x)
#     x   = layers.Dense(64,  activation='relu')(x)
#     x   = layers.Dropout(dropout / 2)(x)
#     out = layers.Dense(1, name='wear_pred')(x)
#     model = Model(inp, out)
#     model.compile(
#         optimizer=keras.optimizers.Adam(lr),
#         loss='mse',
#         metrics=['mae']
#     )
#     return model

# SETS Initialisation

In [30]:
TRAIN_SETS = [1,2,3,4]
EVAL_SETS = [5,6]


# Dataset creation function

- We can't randomly choose the training and validation dataset because we are doing a regression on a sequential time series data. 
- Therefore, we will use masks to mask the dataset based on the sets and divide it using the Sets initialised above

In [31]:
def create_datasets(df, sets, scaler, mode='train', target_col='Wear_Estimate', drop_cols=None):
    """
    Splits the dataframe into perfectly aligned X and y arrays based on set numbers,
    and automatically scales the features.
    """
    if drop_cols is None:
        # Drop identifiers and targets before scaling!
        drop_cols = ['set_no', 'B_load_trend_r2', target_col]
        
    # 1. Create Masks
    mask = df['set_no'].isin(sets)
    sub_df = df[mask].copy()
    
    # 2. Extract Features (X)
    # Make sure we drop the labels before extracting values!
    X = sub_df.drop(columns=drop_cols, errors='ignore')
    # print(X.columns)

    columns_list = list(X.columns)
    if target_col in columns_list:
        raise Exception(f'Target Value Leakage')
    X = X.values
    
    
    # 3. Apply Scaling
    if mode == 'train':
        X = scaler.fit_transform(X)
    else:
        X = scaler.transform(X)
        
    # 4. Extract Targets (y) using the EXACT SAME MASK
    y = sub_df[target_col].values
    
    # Print summary
    print(f"Sets {sets} | X shape: {X.shape} | y shape: {y.shape}")
    
    return X, y


# Standard Scaler

- Scaling all the features 

In [32]:
scaler = StandardScaler()

# 1. Fit the scaler on ALL 6 sets FIRST (this is your source of truth)
X_train_all, y_train_all = create_datasets(df=df_train_features, sets=[1,2,3,4,5,6], scaler=scaler, mode='train')

# 2. Now transform all subsets using the SAME 6-set scaler (mode='test' = only transform, never refit)
X_train, y_train = create_datasets(df=df_train_features, sets=[1,2,3,4], scaler=scaler, mode='test')
X_val, y_val     = create_datasets(df=df_train_features, sets=[5,6], scaler=scaler, mode='test')
X_test, y_test   = create_datasets(df=df_test_features, sets=[1,2,3], scaler=scaler, mode='test')


Sets [1, 2, 3, 4, 5, 6] | X shape: (156, 61) | y shape: (156,)
Sets [1, 2, 3, 4] | X shape: (104, 61) | y shape: (104,)
Sets [5, 6] | X shape: (52, 61) | y shape: (52,)
Sets [1, 2, 3] | X shape: (77, 61) | y shape: (77,)


# Removing unwanted features 
- Feature Importance 

In [33]:
df_train_features.drop(columns = ['Wear_Estimate', 'set_no']).columns

Index(['cut_no', 'Acceleration_X_g_band4_energy', 'Acceleration_X_g_band4_std',
       'Acceleration_X_g_band3_std', 'Acceleration_X_g_band3_energy',
       'Acceleration_Y_g_band5_kurtosis', 'AE_V_band4_energy',
       'AE_V_band4_std', 'Acceleration_X_g_band5_kurtosis',
       'AE_V_band2_energy', 'AE_V_band5_energy', 'AE_V_band3_energy',
       'Acceleration_Y_g_kurtosis', 'AE_V_band2_std', 'AE_V_band5_std',
       'Acceleration_X_g_band2_std', 'AE_V_band1_energy',
       'Acceleration_Z_g_kurtosis', 'mainSpndLoad_std', 'AE_V_band3_std',
       'Y_load_range', 'Acceleration_Z_g_band5_kurtosis',
       'AE_V_band2_kurtosis', 'Acceleration_Y_g_band4_energy',
       'Acceleration_Y_g_band4_std', 'AE_V_band0_energy', 'AE_V_energy',
       'Acceleration_X_g_band5_std', 'mainSpndLoad_rms', 'AE_V_band1_std',
       'Acceleration_Y_g_band5_max', 'AE_V_band5_kurtosis',
       'AE_V_band1_entropy', 'Acceleration_X_g_band5_energy',
       'AE_V_band3_kurtosis', 'Acceleration_Y_g_band5_min',
  

# Feature Importance (Mutual Information Gain Regression)

- Setting the MI threshold as 0.4 and removing any features that are less than the 0.4 threshold

In [34]:
print(f'Train Dataset Shape : {X_train.shape}')
print(f'Test Dataset Shape : {X_test.shape}')
print(f'Validation dataset shape : {X_val.shape}')

Train Dataset Shape : (104, 61)
Test Dataset Shape : (77, 61)
Validation dataset shape : (52, 61)


In [35]:
def generate_model_metrics(y_pred, y_true):
    print(f'RMSE : {mean_squared_error(y_pred, y_true) ** 0.5}')
    print(f'MAPE : {mean_absolute_percentage_error(y_pred, y_true)}')
    print(f'R2 Score : {r2_score(y_true, y_pred)}')

In [36]:
MODEL_RESULTS = {
    'model_name' : [], 
    'model_r2_score' : [], 
    'model_mape' : [], 
    'model_rmse' : []
}

In [37]:
import optuna
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 1000),
        'max_depth':        trial.suggest_int('max_depth', 3, 10),
        'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma':            trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'random_state':     42,
        'verbosity':        0
    }
    
    model = XGBRegressor(**params)
    
    # Train on 1-4, Validate on 5-6
    model.fit(
        X_train, y_train, 
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    
    # Tell Optuna to minimize the Validation Error
    preds = model.predict(X_val)
    return np.sqrt(mean_squared_error(y_val, preds))

print("Starting Optuna optimization...")
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=200, show_progress_bar=True)

print(f"\nBest Validation RMSE: {study.best_value:.3f}")


best_xgb_model = XGBRegressor(**study.best_params, random_state=42)



/Applications/anaconda3/envs/a_star/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Starting Optuna optimization...


Best trial: 180. Best value: 10.0728: 100%|██████████| 200/200 [00:53<00:00,  3.77it/s]


Best Validation RMSE: 10.073


In [38]:
best_xgb_model.fit(X_train_all, y_train_all)
#

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.9395590298666043
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import

In [39]:
predictions = best_xgb_model.predict(X_test)




In [40]:
from sklearn.metrics import mean_absolute_percentage_error

# 1. Calculate the MAPE 
mape = mean_absolute_percentage_error(y_test, predictions)
r2 = r2_score(y_test, predictions)
mse = mean_squared_error(y_test, predictions)
# 2. Multiply by 100 to format it as a percentage
print(f"MAPE (%) : {mape * 100:.3f}%")
print(f'MSE : {mse}')
print(f'r2 score : {r2}')
print(f'RMSE : {mse ** 0.5}')

MODEL_RESULTS['model_name'].append('XGBoost_Model')
MODEL_RESULTS['model_r2_score'].append(r2)
MODEL_RESULTS['model_mape'].append(mape)
MODEL_RESULTS['model_rmse'].append(mse ** 0.5)


MAPE (%) : 20.944%
MSE : 323.7362411628711
r2 score : 0.6879147538110335
RMSE : 17.992671873928874


In [41]:
import optuna
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold

# Hide excessive logging from Optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_rf(trial):
    # 1. Define the search space for Random Forest
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.5, 0.8, 1.0]),
        'random_state': 42,
        'n_jobs': -1  # Use all CPU cores for faster training
    }
    
    # 2. Use KFold to split your ALL training data into 3 rotating chunks
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    rmse_scores = []
    
    for train_idx, val_idx in kf.split(X_train_all):
        # Split the data for this specific fold
        X_tr, X_va = X_train_all[train_idx], X_train_all[val_idx]
        y_tr, y_va = y_train_all[train_idx], y_train_all[val_idx]
        
        # Build and train the model
        model = RandomForestRegressor(**params)
        model.fit(X_tr, y_tr)
        
        # Evaluate on the validation fold
        preds = model.predict(X_va)
        fold_rmse = np.sqrt(mean_squared_error(y_va, preds))
        rmse_scores.append(fold_rmse)
        
    # Return the average RMSE across all 3 folds to ensure it generalizes well!
    return np.mean(rmse_scores)

print("Starting Random Forest Optuna optimization...")
study_rf = optuna.create_study(direction='minimize')
study_rf.optimize(objective_rf, n_trials=100, show_progress_bar=True)

print("\n--- RANDOM FOREST OPTIMIZATION COMPLETE ---")
print(f"Best Validation RMSE: {study_rf.best_value:.3f}")
print("Best Hyperparameters:")
for key, value in study_rf.best_params.items():
    print(f"    {key}: {value}")

# =========================================================
# Train the ultimate model on ALL 6 SETS using the best params!
# =========================================================

rf_model = RandomForestRegressor(**study_rf.best_params, random_state=42)



Starting Random Forest Optuna optimization...


Best trial: 97. Best value: 8.69933: 100%|██████████| 100/100 [01:30<00:00,  1.10it/s]


--- RANDOM FOREST OPTIMIZATION COMPLETE ---
Best Validation RMSE: 8.699
Best Hyperparameters:
    n_estimators: 825
    max_depth: 10
    min_samples_split: 2
    min_samples_leaf: 1
    max_features: 0.8


In [42]:
rf_model.fit(X_train_all, y_train_all)


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",825
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",0.8
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples 

In [43]:
rf_preds = rf_model.predict(X_test)

In [44]:
from sklearn.metrics import mean_absolute_percentage_error

# 1. Calculate the MAPE 
mape = mean_absolute_percentage_error(y_test, rf_preds)
r2 = r2_score(y_test, rf_preds)
mse = mean_squared_error(rf_preds, y_test)
# 2. Multiply by 100 to format it as a percentage
print(f"MAPE (%): {mape * 100:.3f}%")
print(f'MSE : {mse}')
print(f'r2 score : {r2}')
print(f'RMSE : {mse ** 0.5}')

MODEL_RESULTS['model_name'].append('Random Forest Regressor')
MODEL_RESULTS['model_r2_score'].append(r2)
MODEL_RESULTS['model_mape'].append(mape)
MODEL_RESULTS['model_rmse'].append(mse ** 0.5)

MAPE (%): 20.026%
MSE : 416.48250478739953
r2 score : 0.5985063501908601
RMSE : 20.407902998284747


In [45]:
import optuna
import numpy as np
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_cb(trial):
    # 1. Define the search space for CatBoost
    params = {
        'iterations': trial.suggest_int('iterations', 100, 1000),
        'depth': trial.suggest_int('depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'random_seed': 42,
        'verbose': 0  # Keep it quiet during training
    }
    
    # 2. 3-Fold Cross Validation
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    rmse_scores = []
    
    for train_idx, val_idx in kf.split(X_train_all):
        X_tr, X_va = X_train_all[train_idx], X_train_all[val_idx]
        y_tr, y_va = y_train_all[train_idx], y_train_all[val_idx]
        
        # Build and train CatBoost
        model = CatBoostRegressor(**params)
        model.fit(X_tr, y_tr)
        
        # Evaluate
        preds = model.predict(X_va)
        fold_rmse = np.sqrt(mean_squared_error(y_va, preds))
        rmse_scores.append(fold_rmse)
        
    # Return average RMSE across folds
    return np.mean(rmse_scores)

print("Starting CatBoost Optuna optimization...")
study_cb = optuna.create_study(direction='minimize')
study_cb.optimize(objective_cb, n_trials=50, show_progress_bar=True)

print("\n--- CATBOOST OPTIMIZATION COMPLETE ---")
print(f"Best CV RMSE: {study_cb.best_value:.3f}")
for key, value in study_cb.best_params.items():
    print(f"    {key}: {value}")

# =========================================================
# Train the ultimate model on ALL 6 SETS
# =========================================================
best_cb_model = CatBoostRegressor(**study_cb.best_params, random_seed=42, verbose=0)
best_cb_model.fit(X_train_all, y_train_all)

# Predict on the True Test Set
final_preds_cb = best_cb_model.predict(X_test)

mse = mean_squared_error(y_test, final_preds_cb)
mae = mean_absolute_error(y_test, final_preds_cb)
r2  = r2_score(y_test, final_preds_cb)

print("\n--- FINAL TEST RESULTS (CATBOOST) ---")
print(f"RMSE:     {np.sqrt(mse):.3f}")
print(f"MAPE (%): {(mae / np.mean(y_test)) * 100:.3f}%") 
print(f"R²:       {r2:.3f}")


Starting CatBoost Optuna optimization...


Best trial: 24. Best value: 8.4993: 100%|██████████| 50/50 [02:20<00:00,  2.82s/it] 



--- CATBOOST OPTIMIZATION COMPLETE ---
Best CV RMSE: 8.499
    iterations: 815
    depth: 3
    learning_rate: 0.1025676008628006
    l2_leaf_reg: 0.2814365319918844
    subsample: 0.8350577753249924

--- FINAL TEST RESULTS (CATBOOST) ---
RMSE:     21.831
MAPE (%): 20.880%
R²:       0.541


In [46]:
MODEL_RESULTS['model_name'].append('CatBoost')
MODEL_RESULTS['model_mape'].append(mape)
MODEL_RESULTS['model_r2_score'].append(r2)
MODEL_RESULTS['model_mape'].append(mape)

In [47]:
from sklearn.linear_model import ElasticNetCV

# ElasticNetCV automatically finds the best hyperparameters for you using Cross-Validation!
enet = ElasticNetCV(
    l1_ratio=[0.2, 0.3, 0.1, 0.5, 0.7, 0.9, 0.99, 1.0], 
    alphas=[0.0001, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0], 
    cv=2, 
    random_state=42,
    max_iter=10000
)

enet.fit(X_train_all, y_train_all)
final_preds_enet = enet.predict(X_test)

print(f"\n--- ELASTICNET RESULTS ---")
print(f"RMSE:     {np.sqrt(mean_squared_error(y_test, final_preds_enet)):.3f}")
print(f"MAPE (%): {(mean_absolute_error(y_test, final_preds_enet) / np.mean(y_test)) * 100:.3f}%")
print(f"R²:       {r2_score(y_test, final_preds_enet):.3f}")


/Applications/anaconda3/envs/a_star/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.291e+02, tolerance: 1.176e+01
  model = cd_fast.enet_coordinate_descent_gram(
/Applications/anaconda3/envs/a_star/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.771e+02, tolerance: 1.176e+01
  model = cd_fast.enet_coordinate_descent_gram(
/Applications/anaconda3/envs/a_star/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the feat


--- ELASTICNET RESULTS ---
RMSE:     16.744
MAPE (%): 15.954%
R²:       0.730


/Applications/anaconda3/envs/a_star/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.388e+02, tolerance: 1.176e+01
  model = cd_fast.enet_coordinate_descent_gram(
/Applications/anaconda3/envs/a_star/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.074e+02, tolerance: 1.176e+01
  model = cd_fast.enet_coordinate_descent_gram(
/Applications/anaconda3/envs/a_star/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the feat

# Testing only for known cuts

RMSE:     16.979
MAPE (%): 16.213%
R²:       0.722

In [48]:
temp = final_wear_df_testset.copy()

In [49]:
gt_eval_set = df_test_features[df_test_features['cut_no'].isin([1,6,11,16,21,26])].copy()
gt_eval_set

,cut_no,Acceleration_X_g_band4_energy,Acceleration_X_g_band4_std,Acceleration_X_g_band3_std,Acceleration_X_g_band3_energy,Acceleration_Y_g_band5_kurtosis,AE_V_band4_energy,AE_V_band4_std,Acceleration_X_g_band5_kurtosis,AE_V_band2_energy,...,Acceleration_Y_g_band3_std,Acceleration_Z_g_band5_std,Y_load_std,Acceleration_Z_g_band5_entropy,Acceleration_Z_g_band5_energy,Acceleration_Z_g_band2_entropy,Acceleration_Z_g_band5_max,Acceleration_Z_g_max,set_no,Wear_Estimate
0,1,11620.569883,0.503756,0.194909,869.963062,5.295941,10.530223,0.015164,2.955939,156.724858,...,0.531288,11.669768,8.097485,10.803988,1.247114e+07,8.797718,91.204563,55.818723,1,33.000000
5,6,15732.608429,0.615827,0.232544,1121.947660,8.536805,13.759648,0.018212,2.168090,236.528644,...,0.671624,19.023201,5.400347,10.767242,3.002215e+07,8.673576,175.127911,94.268363,1,48.000000
10,11,16688.021382,0.633179,0.247620,1276.741342,8.060747,23.045821,0.023530,1.709691,399.442152,...,0.721027,21.351399,5.657785,10.764654,3.794809e+07,8.646872,193.812804,107.013244,1,67.000000
15,16,18740.033877,0.671639,0.259357,1397.530597,2.954220,22.776023,0.023415,1.294103,340.033691,...,0.714586,20.764215,5.888436,10.829669,3.581803e+07,8.755701,145.560569,89.193038,1,88.000000
20,21,23332.857685,0.749753,0.299052,1856.602364,0.667918,33.293209,0.028321,0.925671,497.054971,...,0.763018,21.081415,6.455516,10.884911,3.689137e+07,8.841719,105.875698,78.298708,1,123.000000
25,26,23506.946639,0.733677,0.292636,1870.154687,1.314156,35.441189,0.028488,1.187778,548.767791,...,0.740357,19.947633,5.835034,10.865074,3.475101e+07,8.837399,112.495618,73.774291,1,127.000000
30,6,14475.259215,0.592308,0.232996,1120.595510,10.642258,11.184162,0.016465,2.154322,197.682871,...,0.700637,17.848552,5.935767,10.765577,2.628464e+07,8.695870,176.967461,99.415030,2,60.000000
35,11,17587.474456,0.634832,0.260948,1486.098106,3.275433,18.072809,0.020350,1.621058,321.175102,...,0.716793,18.035037,6.458660,10.840255,2.838665e+07,8.744467,143.937622,77.753437,2,72.000000
40,16,21128.629070,0.713386,0.291610,1765.484681,0.896897,24.457327,0.024272,1.023338,375.176099,...,0.780271,19.678610,6.482974,10.855508,3.215088e+07,8.825303,97.302417,66.110113,2,94.000000
45,21,25608.673341,0.785459,0.302348,1898.760151,0.908488,25.941159,0.025000,1.191760,400.402850,...,0.770586,19.383882,6.664912,10.845062,3.118863e+07,8.811627,94.042275,72.366177,2,120.000000


In [50]:
cut_data_testset = cut_data_testset.rename(columns={
    'Set_No': 'set_no', 
    'Cut_No': 'cut_no',
    'Measurement': 'measurement'
})

In [51]:
cut_data_testset.head()

,set_no,cut_no,measurement
0,1,1,33.0
1,1,6,48.0
2,1,11,67.0
3,1,16,88.0
4,1,21,123.0


In [52]:
# Instead of .join(), use pd.merge()
gt_test = pd.merge(
    gt_eval_set,
    cut_data_testset,
    on=['set_no', 'cut_no'],
    how='left'
)


In [53]:
gt_test.dropna(inplace = True)

In [54]:
gt_test.head()

,cut_no,Acceleration_X_g_band4_energy,Acceleration_X_g_band4_std,Acceleration_X_g_band3_std,Acceleration_X_g_band3_energy,Acceleration_Y_g_band5_kurtosis,AE_V_band4_energy,AE_V_band4_std,Acceleration_X_g_band5_kurtosis,AE_V_band2_energy,...,Acceleration_Z_g_band5_std,Y_load_std,Acceleration_Z_g_band5_entropy,Acceleration_Z_g_band5_energy,Acceleration_Z_g_band2_entropy,Acceleration_Z_g_band5_max,Acceleration_Z_g_max,set_no,Wear_Estimate,measurement
0,1,11620.569883,0.503756,0.194909,869.963062,5.295941,10.530223,0.015164,2.955939,156.724858,...,11.669768,8.097485,10.803988,1.247114e+07,8.797718,91.204563,55.818723,1,33.0,33.0
1,6,15732.608429,0.615827,0.232544,1121.947660,8.536805,13.759648,0.018212,2.168090,236.528644,...,19.023201,5.400347,10.767242,3.002215e+07,8.673576,175.127911,94.268363,1,48.0,48.0
2,11,16688.021382,0.633179,0.247620,1276.741342,8.060747,23.045821,0.023530,1.709691,399.442152,...,21.351399,5.657785,10.764654,3.794809e+07,8.646872,193.812804,107.013244,1,67.0,67.0
3,16,18740.033877,0.671639,0.259357,1397.530597,2.954220,22.776023,0.023415,1.294103,340.033691,...,20.764215,5.888436,10.829669,3.581803e+07,8.755701,145.560569,89.193038,1,88.0,88.0
4,21,23332.857685,0.749753,0.299052,1856.602364,0.667918,33.293209,0.028321,0.925671,497.054971,...,21.081415,6.455516,10.884911,3.689137e+07,8.841719,105.875698,78.298708,1,123.0,123.0


In [55]:
known_test_target = gt_test['measurement']
gt_test.drop(columns = ['Wear_Estimate', 'measurement', 'set_no'], inplace = True)

In [56]:
test_np = scaler.transform(gt_test)

/Applications/anaconda3/envs/a_star/lib/python3.11/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


In [57]:
k_preds = enet.predict(test_np)

In [58]:
import math 
rmse = math.sqrt(mean_squared_error(known_test_target, k_preds))
r2   = r2_score(known_test_target, k_preds)
mape = mean_absolute_percentage_error(known_test_target, k_preds)

print(f"RMSE:     {rmse:.3f}")
print(f"R²:       {r2:.3f}")
print(f"MAPE (%): {mape * 100:.3f}%")

MODEL_RESULTS['model_name'].append('Elasic NetCV_Known_Cuts')
MODEL_RESULTS['model_mape'].append(mape)
MODEL_RESULTS['model_r2_score'].append(r2)
MODEL_RESULTS['model_mape'].append(mape)

RMSE:     19.160
R²:       0.724
MAPE (%): 17.632%


In [59]:
k_preds_xgb = best_xgb_model.predict(test_np)
generate_model_metrics(k_preds, known_test_target)

RMSE : 19.16001307241077
MAPE : 0.1457084759744686
R2 Score : 0.724236311930458


In [60]:
compare_df = pd.DataFrame(columns = ['xgb_preds', 'enet_preds', 'true_wear', 'cut_no'])

compare_df['xgb_preds'] = k_preds_xgb 
compare_df['enet_preds'] = k_preds
compare_df['true_wear'] = known_test_target.values
compare_df['cut_no'] = gt_test['cut_no'].values

In [61]:
compare_df

,xgb_preds,enet_preds,true_wear,cut_no
0,52.405720,35.228224,33.0,1
1,62.191334,60.058017,48.0,6
2,93.870262,85.406054,67.0,11
3,111.324883,111.890223,88.0,16
4,134.131424,137.446403,123.0,21
5,145.543457,161.552792,127.0,26
6,60.052807,58.807306,60.0,6
7,93.936668,86.739138,72.0,11
8,121.252052,112.581301,94.0,16
9,135.631729,136.722168,120.0,21


In [62]:
import numpy as np

# 1. Calculate the absolute error of each model per row
compare_df['xgb_error']  = (compare_df['xgb_preds']  - compare_df['true_wear']).abs()
compare_df['enet_error'] = (compare_df['enet_preds'] - compare_df['true_wear']).abs()

# 2. See which model wins on each row
compare_df['best_model'] = np.where(
    compare_df['xgb_error'] < compare_df['enet_error'], 'XGBoost', 'ElasticNet'
)

print(compare_df[['xgb_preds', 'enet_preds', 'true_wear', 'xgb_error', 'enet_error', 'best_model']])
print(f"\nXGBoost wins:     {(compare_df['best_model'] == 'XGBoost').sum()} rows")
print(f"ElasticNet wins:  {(compare_df['best_model'] == 'ElasticNet').sum()} rows")

# 3. Calculate overall MAE for each model
xgb_mae  = compare_df['xgb_error'].mean()
enet_mae = compare_df['enet_error'].mean()

print(f"\nXGBoost  MAE: {xgb_mae:.3f}")
print(f"ElasticNet MAE: {enet_mae:.3f}")

# 4. Calculate Inverse Error Weights
# Lower MAE = higher weight (we reward the more accurate model)
total_inv = (1/xgb_mae) + (1/enet_mae)
xgb_weight  = (1/xgb_mae) / total_inv
enet_weight = (1/enet_mae) / total_inv

print(f"\n--- Optimal Ensemble Weights ---")
print(f"XGBoost weight:    {xgb_weight:.3f}")
print(f"ElasticNet weight: {enet_weight:.3f}")

# 5. Create the Weighted Ensemble Predictions
compare_df['ensemble_preds'] = (
    xgb_weight  * compare_df['xgb_preds'] + 
    enet_weight * compare_df['enet_preds']
)

# 6. Compare all three models
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error

for col, name in [('xgb_preds', 'XGBoost'), ('enet_preds', 'ElasticNet'), ('ensemble_preds', 'Ensemble')]:
    mask = compare_df['true_wear'].notna()
    y_true = compare_df.loc[mask, 'true_wear']
    y_pred = compare_df.loc[mask, col]
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    
    print(f"\n{name}:  RMSE={rmse:.3f}  MAPE={mape:.2f}%  R²={r2:.3f}")


     xgb_preds  enet_preds  true_wear  xgb_error  enet_error  best_model
0    52.405720   35.228224       33.0  19.405720    2.228224  ElasticNet
1    62.191334   60.058017       48.0  14.191334   12.058017  ElasticNet
2    93.870262   85.406054       67.0  26.870262   18.406054  ElasticNet
3   111.324883  111.890223       88.0  23.324883   23.890223     XGBoost
4   134.131424  137.446403      123.0  11.131424   14.446403     XGBoost
5   145.543457  161.552792      127.0  18.543457   34.552792     XGBoost
6    60.052807   58.807306       60.0   0.052807    1.192694     XGBoost
7    93.936668   86.739138       72.0  21.936668   14.739138  ElasticNet
8   121.252052  112.581301       94.0  27.252052   18.581301  ElasticNet
9   135.631729  136.722168      120.0  15.631729   16.722168     XGBoost
10  135.946777  159.694370      162.0  26.053223    2.305630  ElasticNet
11   50.225937   32.433451       36.0  14.225937    3.566549  ElasticNet
12   68.890244   62.772592       50.0  18.890244   

In [63]:
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error

# 1. Calculate the absolute error of each model per row
compare_df['xgb_error']  = (compare_df['xgb_preds']  - compare_df['true_wear']).abs()
compare_df['enet_error'] = (compare_df['enet_preds'] - compare_df['true_wear']).abs()

# 2. See which model wins on each row (ground truth oracle)
compare_df['best_model'] = np.where(
    compare_df['xgb_error'] < compare_df['enet_error'], 'XGBoost', 'ElasticNet'
)

print(compare_df[['xgb_preds', 'enet_preds', 'true_wear', 'cut_no', 'xgb_error', 'enet_error', 'best_model']])
print(f"\nXGBoost wins:     {(compare_df['best_model'] == 'XGBoost').sum()} rows")
print(f"ElasticNet wins:  {(compare_df['best_model'] == 'ElasticNet').sum()} rows")

# ── Per-regime breakdown ─────────────────────────────────────────────────────


     xgb_preds  enet_preds  true_wear  cut_no  xgb_error  enet_error  \
0    52.405720   35.228224       33.0       1  19.405720    2.228224   
1    62.191334   60.058017       48.0       6  14.191334   12.058017   
2    93.870262   85.406054       67.0      11  26.870262   18.406054   
3   111.324883  111.890223       88.0      16  23.324883   23.890223   
4   134.131424  137.446403      123.0      21  11.131424   14.446403   
5   145.543457  161.552792      127.0      26  18.543457   34.552792   
6    60.052807   58.807306       60.0       6   0.052807    1.192694   
7    93.936668   86.739138       72.0      11  21.936668   14.739138   
8   121.252052  112.581301       94.0      16  27.252052   18.581301   
9   135.631729  136.722168      120.0      21  15.631729   16.722168   
10  135.946777  159.694370      162.0      26  26.053223    2.305630   
11   50.225937   32.433451       36.0       1  14.225937    3.566549   
12   68.890244   62.772592       50.0       6  18.890244   12.77

In [64]:
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error

# ── Step 1: Per-row absolute errors ──────────────────────────────────────────
compare_df['xgb_error']  = (compare_df['xgb_preds']  - compare_df['true_wear']).abs()
compare_df['enet_error'] = (compare_df['enet_preds'] - compare_df['true_wear']).abs()

# ── Step 2: Oracle best model per row (ground truth comparison) ───────────────
compare_df['best_model'] = np.where(
    compare_df['xgb_error'] < compare_df['enet_error'], 'XGBoost', 'ElasticNet'
)

# ── Step 3: Cut-threshold strategy (cut_no <= 11 → ElasticNet, else → XGBoost) ─
THRESHOLD = 11

compare_df['threshold_preds'] = np.where(
    compare_df['cut_no'] <= THRESHOLD,
    compare_df['enet_preds'],
    compare_df['xgb_preds']
)
compare_df['threshold_model'] = np.where(
    compare_df['cut_no'] <= THRESHOLD, 'ElasticNet', 'XGBoost'
)

# ── Step 4: Original inverse-MAE weighted ensemble ───────────────────────────
xgb_mae  = compare_df['xgb_error'].mean()
enet_mae = compare_df['enet_error'].mean()

total_inv   = (1/xgb_mae) + (1/enet_mae)
xgb_weight  = (1/xgb_mae) / total_inv
enet_weight = (1/enet_mae) / total_inv

compare_df['ensemble_preds'] = (
    xgb_weight  * compare_df['xgb_preds'] +
    enet_weight * compare_df['enet_preds']
)

# ── Step 5: Print per-row comparison ─────────────────────────────────────────
print(compare_df[[
    'cut_no', 'true_wear',
    'xgb_preds', 'enet_preds',
    'xgb_error', 'enet_error',
    'best_model', 'threshold_model', 'threshold_preds'
]].to_string())

# ── Step 6: How often does our threshold rule match the oracle? ───────────────
match_pct = (compare_df['threshold_model'] == compare_df['best_model']).mean() * 100
print(f"\nThreshold rule matches oracle: {match_pct:.1f}% of rows")

# ── Step 7: Per-regime MAE breakdown ─────────────────────────────────────────
for regime, mask in [('Early (cut ≤ 11)', compare_df['cut_no'] <= THRESHOLD),
                     ('Late  (cut > 11)', compare_df['cut_no'] >  THRESHOLD)]:
    sub = compare_df[mask]
    xgb_m  = sub['xgb_error'].mean()
    enet_m = sub['enet_error'].mean()
    thr_m  = (sub['threshold_preds'] - sub['true_wear']).abs().mean()
    print(f"\n{regime}  |  n={len(sub)}")
    print(f"  XGBoost    MAE: {xgb_m:.3f}")
    print(f"  ElasticNet MAE: {enet_m:.3f}")
    print(f"  Threshold  MAE: {thr_m:.3f}  ← selected model")

# ── Step 8: Full-dataset metrics for all four strategies ─────────────────────
print("\n" + "="*60)
print(f"{'Model':<20} {'RMSE':>8} {'MAPE%':>8} {'R²':>8}")
print("="*60)

strategies = [
    ('xgb_preds',       'XGBoost'),
    ('enet_preds',      'ElasticNet'),
    ('ensemble_preds',  'Weighted Ensemble'),
    ('threshold_preds', f'Threshold (≤{THRESHOLD}→Enet)'),
]

mask = compare_df['true_wear'].notna()
y_true = compare_df.loc[mask, 'true_wear']

for col, label in strategies:
    y_pred = compare_df.loc[mask, col]
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    print(f"{label:<20}  {rmse:>8.3f}  {mape:>7.2f}%  {r2:>8.3f}")

print("="*60)
print(f"\nWeighted Ensemble → XGB: {xgb_weight:.3f}, ENet: {enet_weight:.3f}")


    cut_no  true_wear   xgb_preds  enet_preds  xgb_error  enet_error  best_model threshold_model  threshold_preds
0        1       33.0   52.405720   35.228224  19.405720    2.228224  ElasticNet      ElasticNet        35.228224
1        6       48.0   62.191334   60.058017  14.191334   12.058017  ElasticNet      ElasticNet        60.058017
2       11       67.0   93.870262   85.406054  26.870262   18.406054  ElasticNet      ElasticNet        85.406054
3       16       88.0  111.324883  111.890223  23.324883   23.890223     XGBoost         XGBoost       111.324883
4       21      123.0  134.131424  137.446403  11.131424   14.446403     XGBoost         XGBoost       134.131424
5       26      127.0  145.543457  161.552792  18.543457   34.552792     XGBoost         XGBoost       145.543457
6        6       60.0   60.052807   58.807306   0.052807    1.192694     XGBoost      ElasticNet        58.807306
7       11       72.0   93.936668   86.739138  21.936668   14.739138  ElasticNet      El

# Best Model Saving



In [65]:
import joblib

# Save the trained ElasticNet model to a file
joblib.dump(enet, 'saved_model/enet_model.pkl')
print("Model saved to enet_model.pkl")


joblib.dump(best_xgb_model, 'saved_model/xgb_model.pkl')

joblib.dump(scaler, 'saved_model/scaler.pkl')
print("Scaler saved to scaler.pkl")

# Save the top_indices too so you apply the same feature selection later
import numpy as np
joblib.dump(selected_feature_names, 'saved_model/selected_features.pkl')
print("Feature names saved to saved_model/selected_features.pkl")



Model saved to enet_model.pkl
Scaler saved to scaler.pkl
Feature names saved to saved_model/selected_features.pkl
